# TP : Prompt Engineering - Interagir efficacement avec un LLM

**Durée :** 1h
**Objectif :** maîtriser 5 techniques de prompting pour obtenir de meilleures réponses d'un LLM.

## Techniques couvertes

| # | Technique | Idée clé |
|---|---|---|
| 1 | Zero-shot | Demander directement, sans exemple |
| 2 | Few-shot | Donner quelques exemples dans le prompt |
| 3 | Chain-of-Thought (CoT) | Demander un raisonnement étape par étape |
| 4 | Role / System prompt | Cadrer le ton et l'expertise du modèle |
| 5 | Sortie structurée (JSON) | Imposer un format machine-readable |

## Deux modes d'exécution

- **Anthropic API** (recommandé) : crée un fichier `.env` à côté du notebook avec `ANTHROPIC_API_KEY=sk-ant-...`. Modèle utilisé : Claude Haiku 4.5 (rapide, peu coûteux).
- **Mode hors ligne** : si pas de clé API, le notebook bascule automatiquement sur un modèle local (Qwen2.5-1.5B-Instruct, ~3 Go téléchargés au premier appel). Plus lent et moins performant mais 100 % gratuit.


> **Version exercice** : ce TP contient deja 5 exercices marques `# TODO` (un par technique de prompting). Le reste (setup, demonstrations) est deja fourni.
> Installe les dependances une seule fois avec `pip install -r requirements.txt` depuis `cours_ml/todo/` (voir le README de ce dossier). Pour utiliser l'API Claude, cree un fichier `.env` avec `ANTHROPIC_API_KEY=...` dans ce dossier ; sinon un modele local (Qwen2.5-1.5B-Instruct) est utilise automatiquement. Le corrige complet se trouve dans `cours_ml/06_prompt_engineering/tp_prompt_engineering.ipynb`.


---
## 0. Setup


In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # charge .env si présent

USE_ANTHROPIC = bool(os.getenv("ANTHROPIC_API_KEY"))

_anthropic_client = None
_local_tokenizer = None
_local_model = None


def _get_anthropic():
    global _anthropic_client
    if _anthropic_client is None:
        import anthropic
        _anthropic_client = anthropic.Anthropic()
    return _anthropic_client


def _get_local():
    global _local_tokenizer, _local_model
    if _local_model is None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        name = "Qwen/Qwen2.5-1.5B-Instruct"
        print(f"Chargement du modèle local {name} (premier appel : ~3 Go à télécharger)...")
        _local_tokenizer = AutoTokenizer.from_pretrained(name)
        _local_model = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto")
        if torch.backends.mps.is_available():
            _local_model = _local_model.to("mps")
    return _local_tokenizer, _local_model


def complete(user_prompt, system=None, max_tokens=512, temperature=0.0):
    """Envoie un prompt et retourne la réponse texte du LLM.

    - Si ANTHROPIC_API_KEY est défini -> API Claude (rapide, recommandé).
    - Sinon -> modèle local Qwen2.5-1.5B-Instruct (CPU/MPS, plus lent).
    """
    if USE_ANTHROPIC:
        client = _get_anthropic()
        msg = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            temperature=temperature,
            system=system or "Tu es un assistant utile.",
            messages=[{"role": "user", "content": user_prompt}],
        )
        return msg.content[0].text
    else:
        import torch
        tok, mdl = _get_local()
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": user_prompt})
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=(temperature > 0),
                temperature=max(temperature, 1e-5),
                pad_token_id=tok.eos_token_id,
            )
        return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


print(f"Mode actif : {'Anthropic API (Claude Haiku 4.5)' if USE_ANTHROPIC else 'Local (Qwen2.5-1.5B-Instruct)'}")


In [ ]:
# Smoke test : ça doit afficher une salutation
print(complete("Dis bonjour en une phrase, en français."))


---
## 1. Zero-shot prompting (≈ 8 min)

**Idée :** on pose directement la question, sans aucun exemple. C'est le mode par défaut.

**Quand ça marche :** tâches simples, bien documentées sur Internet (les LLMs en ont vu plein pendant leur pré-entraînement).

**Quand ça échoue :** quand on attend un format de sortie précis ou un raisonnement multi-étapes.

### Démonstration : classification de sentiment

Premier essai naïf : on demande sans préciser le format de sortie.


In [ ]:
mauvais = complete("Quel est le sentiment de ce tweet : 'Ce film était une perte de temps.'")
print(mauvais)


**Observation :** la réponse est probablement une phrase ("Le sentiment est négatif..."), pas un label exploitable.

**Solution :** être explicite sur le format de sortie attendu.


In [ ]:
bon = complete(
    "Classifie le sentiment du tweet suivant. Réponds uniquement par un seul mot : "
    "'positif' ou 'négatif'.\n\n"
    "Tweet : 'Ce film était une perte de temps.'\n"
    "Réponse :"
)
print(bon)


### Exercice 1 (≈ 5 min)

Écris un prompt zero-shot qui, étant donné un mail client, renvoie **uniquement** la catégorie parmi :
`facturation`, `technique`, `commercial`, `autre`.

Tester sur : `"Bonjour, ma carte bleue n'est pas acceptée sur votre site depuis hier."`


In [ ]:
mail = "Bonjour, ma carte bleue n'est pas acceptée sur votre site depuis hier."

# TODO : écris ton prompt ci-dessous
prompt = f"..."  # à compléter

print(complete(prompt))


---
## 2. Few-shot prompting (≈ 8 min)

**Idée :** on donne 2-5 exemples (question -> réponse) dans le prompt avant la vraie question. Le modèle imite le format.

**Quand l'utiliser :** quand on veut un format de sortie spécifique ou un style particulier, surtout sur des tâches sortant un peu de l'ordinaire.

### Démonstration : extraction de date au format ISO


In [ ]:
# Sans exemples
zero = complete("Extrais la date au format ISO (YYYY-MM-DD) de : 'Rendez-vous le 14 mars 2026.'")
print("Zero-shot :", zero)
print()

# Avec exemples
few = complete(
    "Extrais la date au format ISO (YYYY-MM-DD) de chaque phrase.\n\n"
    "Phrase : 'Le contrat expire le 3 janvier 2025.'\n"
    "Date : 2025-01-03\n\n"
    "Phrase : 'Réunion prévue pour le 28 juillet 2024.'\n"
    "Date : 2024-07-28\n\n"
    "Phrase : 'Rendez-vous le 14 mars 2026.'\n"
    "Date :"
)
print("Few-shot :", few)


### Exercice 2 (≈ 5 min)

Écris un prompt **few-shot** qui transforme un nom de produit en code SKU au format `CAT-NNN`, où CAT est une abréviation de la catégorie (3 lettres majuscules) et NNN un numéro à 3 chiffres.

Exemples à fournir au modèle :
- `Chaussures de course` -> `CHA-042`
- `Livre de cuisine` -> `LIV-117`
- `Ordinateur portable` -> `ORD-208`

Tester sur : `Casque audio`


In [ ]:
# TODO : construis ton prompt avec les 3 exemples ci-dessus, puis demande pour "Casque audio"
prompt = "..."

print(complete(prompt))


---
## 3. Chain-of-Thought (CoT) (≈ 8 min)

**Idée :** demander explicitement au modèle de **raisonner étape par étape** avant de donner la réponse finale.

**Pourquoi ça marche :** les LLMs prédisent un token à la fois ; leur "réflexion" se passe dans les tokens qu'ils génèrent. En leur laissant de la place pour expliciter chaque étape, on améliore la précision sur les problèmes multi-étapes (maths, logique, déduction).

### Démonstration : problème de mots


In [ ]:
question = (
    "Lucie a 23 pommes. Elle en mange 4 et en donne 7 à sa sœur. "
    "Puis elle achète 3 cagettes de 12 pommes chacune. Combien a-t-elle de pommes maintenant ?"
)

# Sans CoT
direct = complete(question + "\nRéponds uniquement par le nombre.")
print("Réponse directe :", direct)
print()

# Avec CoT
cot = complete(question + "\nRaisonne étape par étape, puis donne le résultat final.")
print("Avec CoT :\n", cot)


### Exercice 3 (≈ 5 min)

Énigme : *"Trois ampoules sont dans une pièce fermée. Trois interrupteurs sont à l'extérieur. Tu peux activer les interrupteurs autant de fois que tu veux, mais tu ne peux entrer qu'une seule fois dans la pièce. Comment identifier quel interrupteur correspond à quelle ampoule ?"*

Écris deux prompts : un qui demande la réponse directe, un qui force le raisonnement étape par étape. Compare les réponses.


In [ ]:
enigme = (
    "Trois ampoules sont dans une pièce fermée. Trois interrupteurs sont à l'extérieur. "
    "Tu peux activer les interrupteurs autant de fois que tu veux, mais tu ne peux entrer "
    "qu'une seule fois dans la pièce. Comment identifier quel interrupteur correspond à quelle ampoule ?"
)

# TODO : prompt direct
print("=== Direct ===")
print(complete("..."))
print()

# TODO : prompt avec CoT
print("=== Avec CoT ===")
print(complete("..."))


---
## 4. Role / System prompt (≈ 8 min)

**Idée :** le `system` prompt cadre le rôle, le ton et l'expertise du modèle. Il est appliqué à toute la conversation, sans être visible côté utilisateur.

**Effets concrets :** style (formel/familier), niveau de détail, vocabulaire, persona.

### Démonstration : même question, deux rôles


In [ ]:
question = "Explique-moi ce qu'est la régularisation L2 (Ridge) en machine learning."

print("=== Rôle : statisticien sénior ===")
print(complete(
    question,
    system="Tu es un statisticien sénior. Tu réponds avec précision et utilises le vocabulaire mathématique exact."
))
print()

print("=== Rôle : prof d'école qui parle à un enfant de 10 ans ===")
print(complete(
    question,
    system="Tu es un prof d'école primaire. Tu expliques tout avec des analogies simples et sans jargon, comme à un enfant de 10 ans."
))


### Exercice 4 (≈ 5 min)

Écris **trois system prompts** pour transformer la même question (`"Faut-il investir en bourse ?"`) en :

1. Une réponse d'un conseiller financier prudent
2. Une réponse d'un trader optimiste
3. Une réponse d'un économiste académique neutre

Lance les 3 versions et compare le ton et les arguments.


In [ ]:
question = "Faut-il investir en bourse ?"

# TODO : 3 system prompts différents
systems = [
    "...",  # conseiller financier prudent
    "...",  # trader optimiste
    "...",  # économiste académique
]

for s in systems:
    print(f"--- {s[:50]} ---")
    print(complete(question, system=s))
    print()


---
## 5. Sortie structurée (JSON) (≈ 10 min)

**Idée :** demander au modèle de répondre dans un format JSON précis, parsable directement en dictionnaire Python.

**Quand l'utiliser :** dès qu'on veut **utiliser** la sortie du LLM dans du code (pipeline, base de données, API).

**Bonnes pratiques :**
1. Décrire le schéma attendu dans le prompt (clés, types).
2. Donner un exemple de sortie JSON.
3. Demander explicitement "Réponds uniquement par du JSON valide, sans texte autour."
4. Parser avec `json.loads` et gérer les erreurs.

### Démonstration : extraction d'entités depuis une bio


In [ ]:
bio = (
    "Marie Curie, née le 7 novembre 1867 à Varsovie, était une physicienne "
    "et chimiste polonaise naturalisée française. Elle a reçu le prix Nobel "
    "de physique en 1903 et celui de chimie en 1911."
)

prompt = f"""Extrais les informations suivantes de la bio ci-dessous et réponds UNIQUEMENT par un objet JSON valide.

Schéma attendu :
{{
  "nom": str,
  "date_naissance": str (YYYY-MM-DD),
  "lieu_naissance": str,
  "nationalites": list[str],
  "prix_nobel": list[{{"annee": int, "discipline": str}}]
}}

Bio :
{bio}

JSON :"""

reponse = complete(prompt, max_tokens=300)
print("Réponse brute :\n", reponse)
print()

# Parsing
try:
    data = json.loads(reponse)
    print("Parsé :", data)
    print("Nom :", data["nom"])
    print("Nombre de prix Nobel :", len(data["prix_nobel"]))
except json.JSONDecodeError as e:
    print("Erreur de parsing :", e)
    print("Astuce : isoler le JSON entre la première '{' et la dernière '}' puis re-parser.")


### Exercice 5 (≈ 8 min)

À partir de la description d'un produit, extrais en JSON :
- `nom` (str)
- `prix_eur` (float)
- `caracteristiques` (list[str], 3 maximum)
- `disponible` (bool)

Description :
> "iPhone 17 Pro Max 256 Go, écran OLED 6.7 pouces, double caméra 48 MP, autonomie 24h. Disponible en stock à 1 449,00 €."

Écris le prompt, parse la sortie avec `json.loads`, et accède au prix en Python.


In [ ]:
description = (
    "iPhone 17 Pro Max 256 Go, écran OLED 6.7 pouces, double caméra 48 MP, "
    "autonomie 24h. Disponible en stock à 1 449,00 €."
)

# TODO : ton prompt
prompt = "..."

reponse = complete(prompt, max_tokens=300)
print("Brut :", reponse)

# TODO : parse et affiche le prix
# data = json.loads(reponse)
# print(f"Prix : {data['prix_eur']} €")


---
## 6. Synthèse (≈ 10 min)

### Comparaison des techniques

| Technique | Quand l'utiliser | Coût additionnel | Gain typique |
|---|---|---|---|
| Zero-shot | Tâches simples, format évident | 0 | Baseline |
| Few-shot | Format de sortie spécifique | + tokens d'exemples | Gros gain sur le respect du format |
| Chain-of-Thought | Problèmes multi-étapes | + tokens de raisonnement | Gros gain en précision |
| Role / System | Adapter ton, niveau, persona | 0 | Améliore la qualité perçue |
| Sortie JSON | Intégration dans un pipeline | + description du schéma | Sortie exploitable en code |

### Règles d'or

1. **Sois spécifique** sur le format attendu (longueur, structure, langue).
2. **Donne des exemples** quand le format est non trivial.
3. **Laisse de la place pour réfléchir** sur les problèmes complexes (CoT).
4. **Cadre le rôle** avec un system prompt si tu veux un ton particulier.
5. **Itère** : compare 2-3 versions d'un prompt, garde la meilleure.
6. **Teste les cas limites** : entrées vides, ambiguës, hostiles (prompt injection).

### Pour aller plus loin

- **Prompt chaining** : décomposer une tâche complexe en plusieurs appels au LLM.
- **RAG** (Retrieval-Augmented Generation) : injecter du contexte récupéré d'une base documentaire.
- **Tool use / function calling** : laisser le LLM appeler des fonctions Python pour calculer, chercher, etc.
- **Évaluation systématique** : mesurer la qualité avec un jeu de tests (LLM-as-a-judge, métriques classiques).
- **Sécurité** : prompt injection, jailbreaks, garde-fous (Anthropic Constitutional AI).


---
## Session à rendre

Cette section est à compléter et à rendre à l'issue du TP. Réponds à chaque question dans la
cellule *Réponse* juste en dessous, à partir des résultats que **tu as toi-même obtenus** en
réalisant ce notebook (il n'y a pas de réponse générique valable pour tout le monde : les valeurs
numériques, choix d'hyperparamètres et graphiques dépendent de ton exécution).

**Q1.** Pour l'exercice 1 (zero-shot), quelle catégorie ton prompt a-t-il renvoyée pour le mail test ? A-t-il fallu itérer pour obtenir une réponse dans un format exploitable (un seul mot) ?

*Réponse :*

_(à compléter)_

**Q2.** Pour l'exercice 2 (few-shot), le modèle a-t-il correctement généré le SKU attendu (format CAT-NNN) pour "Casque audio" ? Si non, quel ajustement au prompt as-tu tenté ?

*Réponse :*

_(à compléter)_

**Q3.** Pour l'exercice 3 (Chain-of-Thought), compare la réponse directe et la réponse avec raisonnement à l'énigme des ampoules : le CoT a-t-il permis d'obtenir une réponse correcte/plus cohérente ?

*Réponse :*

_(à compléter)_

**Q4.** Pour l'exercice 4 (role/system prompt), résume en 2-3 phrases comment le ton et les arguments ont changé entre les 3 personas (conseiller prudent, trader optimiste, économiste).

*Réponse :*

_(à compléter)_

**Q5.** Pour l'exercice 5 (sortie JSON), ton JSON a-t-il été parsable du premier coup avec json.loads ? Si non, qu'as-tu dû faire pour le récupérer ?

*Réponse :*

_(à compléter)_

**Q6.** Sur l'ensemble du TP, quelle technique t'a semblé apporter le gain le plus net par rapport au prompt "naïf" initial, et pour quel type de tâche ?

*Réponse :*

_(à compléter)_